# UniLumos BSS Official-Protocol Smoke

This notebook prepares a one-case UniLumos BSS smoke run for Colab. It does not run the model unless `RUN_SMOKE` is set to `True`.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

IN_COLAB

In [ ]:
TASK_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Colab_Projects/Lumos-Custom/UniLumos'),
    Path('/content/Lumos-Custom/UniLumos'),
    Path.cwd(),
]
TASK_ROOT = next((p for p in TASK_ROOT_CANDIDATES if (p / 'UniLumos' / 'unilumos_infer_abc.py').exists()), TASK_ROOT_CANDIDATES[0])
CODE_ROOT = TASK_ROOT / 'UniLumos'
WEIGHTS_ROOT = Path('/content/drive/MyDrive/Colab_Projects/UniLumos/weights')
EXP_ROOT = Path('/content/drive/MyDrive/Colab_Projects/UniLumos-BSS-Runs/model_b_unilumos_official_bss_smoke_v1')

os.environ['PYTHONPATH'] = f"{TASK_ROOT}:{CODE_ROOT}:" + os.environ.get('PYTHONPATH', '')
print('TASK_ROOT =', TASK_ROOT)
print('CODE_ROOT =', CODE_ROOT)
print('WEIGHTS_ROOT =', WEIGHTS_ROOT)
print('EXP_ROOT =', EXP_ROOT)

In [ ]:
required = [
    CODE_ROOT / 'examples' / 'examples_refined.csv',
    CODE_ROOT / 'unilumos_infer_abc.py',
    CODE_ROOT / 'src' / 'schedulers' / 'RFLOW_WANX21_T2V.py',
]
weights = [
    WEIGHTS_ROOT / 'models_t5_umt5-xxl-enc-bf16.pth',
    WEIGHTS_ROOT / 'umt5-xxl',
    WEIGHTS_ROOT / 'vae.pth',
    WEIGHTS_ROOT / 'unilumos.pt',
]
for p in required + weights:
    print(('OK      ' if p.exists() else 'MISSING '), p)

In [ ]:
from bss_experiments.unilumos.bss_core.grids import get_uniform_shifted_sigmas, make_boundary_split_coords
from bss_experiments.unilumos.bss_core.validate import validate_schedule_payload

base8 = get_uniform_shifted_sigmas(8, 8.0)
bss10, meta = make_boundary_split_coords(base8)
payload = {
    'model_name': 'UniLumos', 'method': 'bss10', 'sampler_mode': 'bss',
    'sample_steps': 10, 'sample_shift': 8.0, 'base_sample_steps': 8,
    'actual_nfe': 10, 'split_pairs': '0,-1', 'terminal_coord': 0.0,
    'base_sigmas': base8, 'final_sigmas': bss10, 'timesteps': list(range(10)),
}
validate_schedule_payload(payload)

In [ ]:
manifest_cmd = [
    sys.executable,
    str(TASK_ROOT / 'bss_experiments' / 'unilumos' / 'scripts' / 'make_manifest_unilumos_official_smoke.py'),
    '--experiment_root', str(EXP_ROOT),
    '--max_cases', '1',
    '--mode', 'abc',
]
manifest_path = subprocess.check_output(manifest_cmd, text=True).strip()
manifest_path

In [ ]:
dry_run_cmd = [
    sys.executable,
    str(TASK_ROOT / 'bss_experiments' / 'unilumos' / 'scripts' / 'run_manifest.py'),
    '--manifest', manifest_path,
    '--weights_root', str(WEIGHTS_ROOT),
    '--dry_run',
]
subprocess.check_call(dry_run_cmd)

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    run_cmd = [
        sys.executable,
        str(TASK_ROOT / 'bss_experiments' / 'unilumos' / 'scripts' / 'run_manifest.py'),
        '--manifest', manifest_path,
        '--weights_root', str(WEIGHTS_ROOT),
        '--resume',
    ]
    subprocess.check_call(run_cmd)

In [ ]:
if RUN_SMOKE:
    schedule_paths = sorted((EXP_ROOT / 'schedules').glob('*_schedule.json'))
    validate_cmd = [sys.executable, str(TASK_ROOT / 'bss_experiments' / 'unilumos' / 'scripts' / 'validate_schedules.py'), *map(str, schedule_paths)]
    subprocess.check_call(validate_cmd)

In [ ]:
if RUN_SMOKE:
    metrics_cmd = [
        sys.executable,
        str(TASK_ROOT / 'bss_experiments' / 'unilumos' / 'scripts' / 'compute_metrics_against_ref.py'),
        '--manifest', manifest_path,
    ]
    metrics_csv = subprocess.check_output(metrics_cmd, text=True).strip()
    table_cmd = [
        sys.executable,
        str(TASK_ROOT / 'bss_experiments' / 'unilumos' / 'scripts' / 'make_main_table.py'),
        '--metrics_csv', metrics_csv,
    ]
    subprocess.check_call(table_cmd)